Simulated Annealing Experiments

In [56]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [57]:
from datetime import datetime

# Parameters
results_path = f'data/simulated_annealing/summary_{datetime.now().strftime("%Y-%m-%d_%H-%M-%S")}.npy'
brute_force_data_directory = './data/brute_force/first_run'

num_runs = 1000

# I think init and end temperatures should be chosen dependent on the delta of the energy,
# i.e. delta = new_loss - old_loss (in the exponent)
# larger delta -> larger temperature (and conversely smaller beta)
init_temp = 1000
end_temp = 1
max_temp_iterations = 101

In [ ]:
import numpy as np

np.random.seed(4321) # set global seed for reproducibility

In [58]:
from pathlib import Path
import pandas as pd
import re

extract_candidate_dimension = lambda p: np.log2(int(re.search(string=str(p.name), pattern=r'_(\d*)_candidates').group(1)))
extract_load_factor = lambda p: float(re.search(string=str(p.name), pattern=r'_(\d*\.\d*)_load_factor').group(1))

data_files = [p for p in Path(brute_force_data_directory).glob("*.csv") if p.is_file()]
candidate_dims = [(extract_candidate_dimension(p), extract_load_factor(p), p) for p in data_files]

df = pd.DataFrame(data=candidate_dims, columns=['dim', 'load factor', 'path'])
df = df.groupby(['dim', 'load factor'])['path'].agg(list).reset_index()

In [59]:
problem_sizes = df['dim'].unique()
load_factors = df['load factor'].unique()
brute_force_data_paths = np.array(df['path'].to_list(), dtype=object).reshape(len(problem_sizes), len(load_factors), -1)

problem_sizes = problem_sizes

In [60]:
problem_sizes.shape

(19,)

In [61]:
brute_force_data_paths.shape

(19, 10, 10)

In [62]:
import pandas as pd
import numpy as np
brute_force_data = np.vectorize(pd.read_csv)(brute_force_data_paths)
loss_spaces = [np.array([d['loss'] for d in da]) for data in brute_force_data for da in data] # needs to be list because is inhomogeneous

In [63]:
from cl_optimizer import SimulatedAnnealing

def create_simulated_annealing_instances(x: np.ndarray):
    return np.array([[
        SimulatedAnnealing(
            lookup_table=data_path,
            bounds=int(x[0]),
        ) for _ in range(num_runs)
    ] for data_path in x[1:]])

sim_annss = np.swapaxes(np.array([np.apply_along_axis(func1d=create_simulated_annealing_instances, arr=np.column_stack((problem_sizes, brute_force_2d)), axis=1) for brute_force_2d in np.swapaxes(brute_force_data, 0, 1)]), 0, 1) # a sim_ann per problem size, instance and load factor

In [64]:
sim_annss.shape

(19, 10, 10, 2)

In [65]:
from collections import defaultdict
from joblib import Parallel, delayed
from parallelization import construct_parallelization_arg_list, get_execute_sim_ann

resultssss = []
for i, sas_per_problem_size in enumerate(sim_annss):
    resultsss = []
    for j, sas_per_problem_instance in enumerate(sas_per_problem_size):
        resultss = []
        for sas_per_load_factor in sas_per_problem_instance:
            arg_list = construct_parallelization_arg_list(sas_per_load_factor, depth=max_temp_iterations, step=10, batch_size=8)

            print(f"Problem size: {i + 1}/{int(problem_sizes[-1])} - instances: {j + 1}/{len(sas_per_problem_instance)}")
            print(f"Preparing to execute {len(arg_list)} in parallel.")
            results = Parallel(n_jobs=-1, verbose=11)(
                delayed(get_execute_sim_ann(init_temp, end_temp))(s_id, sim_ann, temp_iterations) for (s_id, sim_ann), temp_iterations in arg_list)

            groups = defaultdict(list)
            for key, arr in results:
                groups[key].append(arr)

            # Column stack all arrays with the same key
            results = np.array([np.column_stack(groups[k]) for k in sorted(groups)])

            resultss.append(results)
        resultsss.append(resultss)
    resultssss.append(resultsss)

Problem size: 1/19 - instances: 1/10
Preparing to execute 4 in parallel.


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   1 tasks      | elapsed:    0.7s
[Parallel(n_jobs=4)]: Done   2 out of   4 | elapsed:    0.7s remaining:    0.7s
[Parallel(n_jobs=4)]: Done   4 out of   4 | elapsed:    0.7s finished
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Batch computation too fast (0.05501294136047363s.) Setting batch_size=2.
[Parallel(n_jobs=4)]: Done   2 out of   4 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=4)]: Done   4 out of   4 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Batch computation too fast (0.05050063133239746s.) Setting batch_size=2.
[Parallel(n_jobs=4)]: Done   2 out of   4 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=4)]: 

Problem size: 1/19 - instances: 1/10
Preparing to execute 4 in parallel.
Problem size: 1/19 - instances: 1/10
Preparing to execute 4 in parallel.
Problem size: 1/19 - instances: 1/10
Preparing to execute 4 in parallel.


[Parallel(n_jobs=4)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Batch computation too fast (0.054106950759887695s.) Setting batch_size=2.
[Parallel(n_jobs=4)]: Done   2 out of   4 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=4)]: Done   4 out of   4 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Batch computation too fast (0.05300188064575195s.) Setting batch_size=2.
[Parallel(n_jobs=4)]: Done   2 out of   4 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=4)]: Done   4 out of   4 | elapsed:    0.0s finished
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   1 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Batch computation too fast (0.05538463592529297s.) Setting batch_size=2.
[Parallel(n_jobs=4)]: Done   2 out of   4 | elapsed:    0.0s remaining:    0.0s
[

Problem size: 1/19 - instances: 1/10
Preparing to execute 4 in parallel.
Problem size: 1/19 - instances: 1/10
Preparing to execute 4 in parallel.
Problem size: 1/19 - instances: 1/10
Preparing to execute 4 in parallel.


...

[Parallel(n_jobs=4)]: Done   1 tasks      | elapsed:    8.0s
[Parallel(n_jobs=4)]: Done   2 out of   4 | elapsed:    8.2s remaining:    8.2s
[Parallel(n_jobs=4)]: Done   4 out of   4 | elapsed:    9.3s finished
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.


Problem size: 19/19 - instances: 10/10
Preparing to execute 4 in parallel.


[Parallel(n_jobs=4)]: Done   1 tasks      | elapsed:    7.8s
[Parallel(n_jobs=4)]: Done   2 out of   4 | elapsed:    8.1s remaining:    8.1s
[Parallel(n_jobs=4)]: Done   4 out of   4 | elapsed:    9.2s finished
[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.


Problem size: 19/19 - instances: 10/10
Preparing to execute 4 in parallel.


[Parallel(n_jobs=4)]: Done   1 tasks      | elapsed:    7.7s
[Parallel(n_jobs=4)]: Done   2 out of   4 | elapsed:    8.2s remaining:    8.2s
[Parallel(n_jobs=4)]: Done   4 out of   4 | elapsed:    9.2s finished


In [66]:
result_tensor = np.stack(resultssss, axis=0).squeeze()

In [67]:
indices = np.argsort(result_tensor[..., 0], axis=3) # sort by temp. iterations (parallelization does not guarantee order) along runs (axis=3)
indices_expanded = np.expand_dims(indices, axis=-1)
result_tensor = np.take_along_axis(result_tensor, indices_expanded, axis=3)

In [68]:
result_tensor.shape # should be a 6-tuple: (n_problem_sizes, n_instances_per_problem_size, n_load_factors_per_instance, n_runs_per_load_factor, n_temp_iter_steps, 2)

(19, 10, 10, 2, 11, 2)

In [69]:
np.save(results_path, result_tensor)